# Phase 10: our own lightning LoRAs, 8 steps and 4 steps

**Base:** version 1, the unhealed 30-block student (B + block-5 attention bridge step 200 + block-20 attention
bridge step 600). The teacher samples in 40 steps without classifier-free guidance; the goal is 8 steps (about 5x
faster) and 4 steps (about 10x), keeping spelling. Reference points: `phase9-lightning-baseline.ipynb`.

**Method: trajectory distillation.** The intact teacher's 40-step trajectories are recorded (latent at every step).
The student's K steps use exact subsets of the teacher's 40-step sigma grid (every 5th point for 8 steps, every 10th
for 4). Training starts the student at a teacher latent and teaches one student step to land where the teacher's 5
(or 10) steps land: the target velocity is (x_next - x_now) / (sigma_next - sigma_now) along the teacher path. A
LoRA on the attention of the 30 original blocks learns it; the bridges stay frozen. Every checkpoint is saved and
judged quantitatively (distance of the K-step final latent from the teacher's 40-step one, on 45 held-out prompts)
and qualitatively (the four-seed text grids and the 25 general prompts).

**Steps:** (1) record the teacher trajectories; (2) train the 8-step LoRA; (3) train the 4-step LoRA; (4) choose
and fuse.

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
assert (ROOT / 'PLAN.md').exists(), 'Run this notebook from the repository root.'
sys.path.insert(0, str(ROOT / 'tools'))
for stream in (sys.stdout, sys.stderr):      # Modal prints non-ASCII progress symbols
    if hasattr(stream, 'reconfigure'):
        stream.reconfigure(encoding='utf-8')
ALLOW_GPU = False          # set True only to launch a step whose saved result is missing
SOURCES = {}               # worker sources, embedded per step exactly as executed


def load_summary(run):
    path = ROOT / run / 'summary.json'
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None


def reuse_or_launch(run, launch):
    summary = load_summary(run)
    if summary is not None:
        print('Saved result:', run, '| status', summary.get('status'))
        return summary
    if (ROOT / run).parent.joinpath('aborted.json').exists():
        print('Run stopped by request; nothing to reuse.')
        return None
    if not ALLOW_GPU:
        raise RuntimeError(f'No saved result for {run}; set ALLOW_GPU = True to launch it on Modal.')
    launch()
    return load_summary(run)


def pct(value, digits=2):
    return f'{100 * value:.{digits}f}%'


def velocity_row(label, block):
    o = block['overall']['teacher_relative_l2']
    p = {k: block['panels'][k]['teacher_relative_l2']['mean'] for k in block['panels']}
    s = {k: block['stages'][k]['teacher_relative_l2']['mean'] for k in block['stages']}
    return (f"| {label} | {pct(o['mean'])} | {pct(p['gate10'])} | {pct(p['wider50'])} | "
            f"{pct(s['early'], 1)} / {pct(s['middle'], 1)} / {pct(s['late'], 1)} | {pct(o['p90'], 1)} | {pct(o['maximum'], 1)} |")


def velocity_table(rows):
    head = '| Model | Overall | Check (10) | Wider (50) | Early / middle / late | p90 | Worst |\n|---|---|---|---|---|---|---|\n'
    display(Markdown(head + '\n'.join(rows)))


def show_variants(summary):
    rows = [velocity_row(f"{name} (step {v['fit']['selected_step']}, {v['parameters'] / 1e6:.1f}M params)", v)
            for name, v in summary['variants'].items()]
    velocity_table(rows)


def show_images(folder, names):
    for name in names:
        display(Image(filename=str(ROOT / folder / name), width=1000))

## 1. Teacher trajectories

The intact teacher (32 blocks, 40 steps) on the 500 training prompts of the Phase 7 recordings (450 text, 50 general,
same seeds and sizes) and on the 45 held-out image prompts (5 text prompts with 4 seeds, 25 general prompts; checked
not to overlap). Saved per prompt: the 41 latents (start noise to final), the 41 sigmas, and the prompt conditioning.

In [ ]:
# Teacher trajectories: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.
SOURCES['teacher_trajectories_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/teacher_trajectories') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['trajectory']
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # The intact 32-block teacher, 40 steps as always; a forward pre-hook records the latent and timestep the transformer
        # receives at every step, and the conditioning of the first step; the step callback records the final latent.
        outputs = {v: {} for v in variants}
        state = {}

        def capture(module, args, kwargs):
            state['steps'].append((kwargs['hidden_states'].detach().to('cpu').clone(), kwargs['timestep'].detach().float().flatten()[0].item()))
            if not state['conditioning']:
                for field in ('encoder_hidden_states', 'encoder_hidden_states_mask', 'img_shapes', 'img_mask'):
                    value = kwargs[field]
                    state['conditioning'][field] = value.detach().to('cpu').clone() if torch.is_tensor(value) else value

        def on_step_end(pipeline, index, timestep, tensors):
            if index == request['teacher_steps'] - 1:
                state['final'] = tensors['latents'].detach().to('cpu').clone()
            return {}

        handle = model.register_forward_pre_hook(capture, with_kwargs=True)
        model.eval()
        for row in request['rows']:
            path = directory / (row['source_id'] + '-trajectory.pt')
            state.update(steps=[], conditioning={}, final=None)
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=request['teacher_steps'],
                     true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                     use_kv_cache=True, output_type='latent', callback_on_step_end=on_step_end)
            torch.cuda.synchronize()
            sigmas = pipe.scheduler.sigmas.detach().float().cpu().clone()
            if len(state['steps']) != request['teacher_steps'] or state['final'] is None or len(sigmas) != request['teacher_steps'] + 1:
                raise RuntimeError('Trajectory length differs: ' + row['source_id'])
            # The pipeline passes (sigma * 1000) in BF16, divided by 1000 in BF16: reproduce that rounding exactly.
            expected = (pipe.scheduler.timesteps.detach().cpu().to(state['steps'][0][0].dtype) / 1000).float().tolist()
            timesteps = [t for _, t in state['steps']]
            if max(abs(t - e) for t, e in zip(timesteps, expected)) > 1e-6 or max(abs(t - s) for t, s in zip(timesteps, sigmas.tolist())) > 1e-2:
                raise RuntimeError('Recorded timesteps differ from the scheduler: ' + row['source_id'])
            latents = torch.stack([h[0] for h, _ in state['steps']] + [state['final'][0]])
            temporary = path.with_suffix('.tmp')
            torch.save({'request_key': key, 'row': row, 'sigmas': sigmas, 'timesteps': timesteps, 'latents': latents,
                        'conditioning': state['conditioning']}, temporary)
            temporary.replace(path)
            outputs['trajectory'][row['source_id']] = {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size,
                                                       'seconds': time.perf_counter() - started}
            report['calls']['trajectory'] += 1
            if report['calls']['trajectory'] % 25 == 0:
                (directory / 'progress.json').write_text(json.dumps(report, indent=2))
                results_volume.commit()
        handle.remove()
        results_volume.commit()
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    base = json.loads(next((root / 'data/recovered/modal/phase7/candidate_images').glob('*/request.json')).read_bytes())['request']
    text8 = load('data/phase7/block13_bridge_e2e_text8/ff9d259ae589bdf15b98076930c4d16340ef41f4a2c3db897ea3167e95397054/request.json')['request']
    # Held out: the 5 text prompts with 4 seeds and the 25 general prompts used for every image test.
    text_rows = [r for r in base['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    held_out = list(text_rows) + [{**r, 'source_id': f"{r['source_id']}-s{k}", 'seed': (r['seed'] + 1000003 * k) % 2**32}
                                  for r in text_rows for k in (1, 2, 3)]
    held_out += [r for r in base['rows'] if r['subdimension'] not in ('text_rendering', 'posters')]
    train = [dict(r, split='train') for r in text8['new_prompts']]
    if {r['prompt'] for r in train} & {r['prompt'] for r in held_out}:
        raise RuntimeError('Training prompts overlap the held-out image prompts.')
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase10-teacher-trajectories-v1', worker_source=SOURCES['teacher_trajectories_worker.py'],
        execution_account='new', candidate_only=True, teacher_steps=40,
        rows=[dict(r, split='held_out') for r in held_out] + train,
        scope='Intact teacher, 40 steps: latent at every step for 500 training prompts (450 text) and the 45 held-out image prompts')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/teacher_trajectories' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/teacher_trajectories/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase10-teacher-trajectories')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'teacher_trajectories_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN = 'data/phase10/teacher_trajectories/' + main()   # writes the request and returns its key; no GPU
LAUNCH_ARGS = []
summary = reuse_or_launch(RUN, main)
if summary and summary.get('status') == 'passed':
    import statistics
    seconds = [r['trajectory']['seconds'] for r in summary['rows']]
    print(len(seconds), 'trajectories; median seconds', round(statistics.median(seconds), 1),
          '; total GB', round(sum(r['trajectory']['bytes'] for r in summary['rows']) / 1e9, 1))

## 2. The 8-step lightning LoRA

LoRA rank 32 on q, k, v and out of the 30 original blocks (bridges frozen), 2000 updates at lr 1e-4. Every 500 updates:
saved, the endpoint distance on the 45 held-out prompts (quantitative), and the same 45 images (qualitative). Step 0 is
version 1 sampled naively in 8 steps (`data/phase10/lightning_lora/protocol.json`).

In [ ]:
# Lightning LoRA, 8 steps: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run step 1 first).
STUDENT_STEPS, BRIDGE_DTYPE = 8, 'bfloat16'
SOURCES['lightning_lora_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/lightning_lora') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [f'step_{s:04d}' for s in range(0, request['fit']['updates'] + 1, request['fit']['checkpoint_every'])]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # ---- Version 1 student with a trainable LoRA; trajectory distillation to K steps; every checkpoint evaluated. ----
        import random
        from diffusers import FlowMatchEulerDiscreteScheduler
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        from torch.nn.attention import SDPBackend, sdpa_kernel
        K = request['student_steps']
        stride = request['teacher_steps'] // K
        keep = list(range(0, request['teacher_steps'] + 1, stride))
        fit, lora = request['fit'], request['lora']
        outputs = {v: {} for v in variants}
        conditioned, components = {}, {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        bridge_slots = []
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(),
                                                     keep_dtype=request['bridge_dtype'] == 'float32')
            bridge_slots.append(bridge_spec['layer'])
        torch.manual_seed(fit['init_seed'])
        for slot, block in enumerate(list(model.transformer_blocks)):
            if slot in bridge_slots:
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), lora['rank'], lora['alpha']))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], lora['rank'], lora['alpha'])
            model.transformer_blocks[slot] = components['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        model.requires_grad_(False)
        model.gradient_checkpointing = False
        trainable = {n: p for n, p in model.named_parameters() if '.attn.' in n and ('.down.' in n or '.up.' in n)}
        if len(trainable) != 30 * 4 * 2:
            raise RuntimeError('Adapter structure differs.')
        for parameter in trainable.values():
            parameter.requires_grad_(True)
        report['trainable_parameters'] = sum(p.numel() for p in trainable.values())
        gc.collect()
        torch.cuda.empty_cache()
        model.eval()

        # Teacher trajectories: only the K+1 points on the student's grid are kept in memory.
        traj_dir = Path(request['trajectories']['remote_directory'])

        def load_trajectory(source_id):
            path = traj_dir / (source_id + '-trajectory.pt')
            if digest(path) != request['trajectories']['sha256'][source_id]:
                raise RuntimeError('Trajectory differs: ' + source_id)
            return torch.load(path, weights_only=True, map_location='cpu')

        train_items = []
        for source_id in request['trajectories']['train_ids']:
            item = load_trajectory(source_id)
            train_items.append({'source_id': source_id, 'latents': item['latents'][keep].contiguous(), 'sigmas': item['sigmas'][keep],
                                'timesteps': [item['timesteps'][i] for i in keep[:-1]], 'conditioning': item['conditioning']})
        held_final, grid_sigmas = {}, None
        for row in request['rows']:
            item = load_trajectory(row['source_id'])
            held_final[row['source_id']] = item['latents'][-1].float()
            grid_sigmas = item['sigmas'][keep]
        report['train_items'] = len(train_items)

        def train_loss(item, segment):
            # One student step from the teacher's latent must land where the teacher's `stride` steps land.
            x = item['latents'][segment].unsqueeze(0).to('cuda')
            x_next = item['latents'][segment + 1].unsqueeze(0).to('cuda')
            sigma, sigma_next = item['sigmas'][segment].item(), item['sigmas'][segment + 1].item()
            target = (x_next.float() - x.float()) / (sigma_next - sigma)
            cond = item['conditioning']
            prediction = model(hidden_states=x, timestep=torch.tensor([item['timesteps'][segment]], device='cuda', dtype=x.dtype),
                               encoder_hidden_states=cond['encoder_hidden_states'].to('cuda'),
                               encoder_hidden_states_mask=cond['encoder_hidden_states_mask'].to('cuda'),
                               img_shapes=cond['img_shapes'], img_mask=cond['img_mask'].to('cuda'), attention_kwargs=None,
                               kv_cache=QwenImage21KVCache(len(model.transformer_blocks)), kv_cache_mode='extract',
                               return_dict=False)[0][:, -x.shape[1]:].float()
            loss = (prediction - target).square().mean() / target.square().mean().clamp_min(1e-12)
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss.')
            return loss

        class SubsetScheduler(FlowMatchEulerDiscreteScheduler):
            # The teacher's 40-step sigma grid (same dynamic shift and terminal stretch), keeping every `stride`-th point.
            stride = 1

            def set_timesteps(self, *args, **kwargs):
                super().set_timesteps(*args, **kwargs)
                self.sigmas = self.sigmas[::self.stride].clone()
                self.timesteps = (self.sigmas[:-1] * self.config.num_train_timesteps).to(self.sigmas.device)
                self.num_inference_steps = len(self.timesteps)

        default_scheduler = pipe.scheduler
        student_scheduler = SubsetScheduler.from_config(default_scheduler.config)
        student_scheduler.stride = stride
        report['checkpoints'] = []
        report['losses'] = []
        started = time.perf_counter()

        def evaluate(step):
            # Quantitative: K-step final latent against the teacher's 40-step final latent (same noise), 45 held-out prompts.
            # Qualitative: the same 45 images (four-seed text grids and 25 general prompts).
            name = f'step_{step:04d}'
            pipe.scheduler = student_scheduler
            errors = {}
            for row in request['rows']:
                final = {}

                def on_step_end(pipeline, index, timestep, tensors):
                    if index == K - 1:
                        final['latents'] = tensors['latents'].detach().float().cpu()
                    return {}

                torch.cuda.synchronize()
                t0 = time.perf_counter()
                with torch.inference_mode():
                    image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=request['teacher_steps'],
                                 true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                                 use_kv_cache=True, output_type='pil', callback_on_step_end=on_step_end).images[0]
                torch.cuda.synchronize()
                seconds = time.perf_counter() - t0
                if (pipe.scheduler.sigmas.detach().float().cpu() - grid_sigmas).abs().max() > 1e-6:
                    raise RuntimeError('Student sigma grid differs from the teacher subset.')
                reference = held_final[row['source_id']]
                error = ((final['latents'][0] - reference).norm() / reference.norm()).item()
                errors[row['source_id']] = error
                path = directory / (row['source_id'] + '-' + name + '.png')
                image.save(path, format='PNG')
                outputs[name][row['source_id']] = {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size,
                                                   'seconds': seconds, 'endpoint_relative_l2': error}
                report['calls'][name] += 1
            pipe.scheduler = default_scheduler
            text_ids = {r['source_id'] for r in request['rows'] if r['subdimension'] in ('text_rendering', 'posters')}
            mean = lambda values: sum(values) / len(values)
            entry = dict(step=step, seconds=time.perf_counter() - started,
                         endpoint_mean=mean(list(errors.values())),
                         endpoint_text=mean([e for s, e in errors.items() if s in text_ids]),
                         endpoint_general=mean([e for s, e in errors.items() if s not in text_ids]))
            if step > 0:
                path = directory / f'lora_{K}steps_step_{step:04d}.pt'
                temporary = path.with_suffix('.tmp')
                torch.save({'request_key': key, 'step': step, 'student_steps': K, 'lora': lora, 'bridge_dtype': request['bridge_dtype'],
                            'state_dict': {n: p.detach().cpu().clone() for n, p in trainable.items()}}, temporary)
                temporary.replace(path)
                entry['checkpoint'] = dict(name=path.name, sha256=digest(path), bytes=path.stat().st_size)
                report['artifacts'].append(entry['checkpoint'])
            report['checkpoints'].append(entry)
            print(f"{K}-step LoRA, step {step}: endpoint {entry['endpoint_mean']:.4f} (text {entry['endpoint_text']:.4f}, "
                  f"general {entry['endpoint_general']:.4f}) ({entry['seconds']:.0f}s)", flush=True)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()

        optimizer = torch.optim.AdamW(list(trainable.values()), lr=fit['learning_rate'], weight_decay=0.0)
        rng = random.Random(fit['sampling_seed'])
        running = []
        for step in range(fit['updates'] + 1):
            if step % fit['checkpoint_every'] == 0:
                model.eval()
                evaluate(step)
            if step == fit['updates']:
                break
            for group in optimizer.param_groups:
                group['lr'] = fit['learning_rate'] * min(1.0, (step + 1) / fit['warmup'])
            optimizer.zero_grad(set_to_none=True)
            # Backward recomputes checkpointed blocks: keep the forward attention kernel and cache context.
            with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for _ in range(fit['accumulation']):
                    loss = train_loss(rng.choice(train_items), rng.randrange(K)) / fit['accumulation']
                    loss.backward()
                    running.append(loss.item() * fit['accumulation'])
            torch.nn.utils.clip_grad_norm_(list(trainable.values()), fit['gradient_clip'])
            optimizer.step()
            report['optimizer_updates'] += 1
            if len(running) >= 50:
                report['losses'].append({'step': step + 1, 'mean_loss': sum(running) / len(running)})
                running = []
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_train():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    traj = load(RUN + '/summary.json')
    traj_request = load(RUN + '/request.json')['request']
    healing = load(next((root / 'data/phase8/healing').glob('*/request.json')).relative_to(root))['request']
    shas = {r['source_id']: r['trajectory']['sha256'] for r in traj['rows']}
    request = {k: traj_request[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                            'canonical_tables_key', 'core_source', 'wrapper_source', 'teacher_steps')}
    request.update(
        version='phase10-lightning-lora-v1', worker_source=SOURCES['lightning_lora_worker.py'], execution_account='new', candidate_only=True,
        student_steps=STUDENT_STEPS, bridge_dtype=BRIDGE_DTYPE,
        conditioned_bridge_source=healing['conditioned_bridge_source'], components_source=healing['components_source'],
        bridges=healing['bridges'],
        trajectories=dict(remote_directory=f"/phase3/phase10/teacher_trajectories/{traj['request_key']}",
                          train_ids=[r['source_id'] for r in traj_request['rows'] if r['split'] == 'train'], sha256=shas),
        rows=[r for r in traj_request['rows'] if r['split'] == 'held_out'],
        lora=dict(rank=32, alpha=32, targets=['to_q', 'to_k', 'to_v', 'to_out.0']),
        fit=dict(init_seed=20261019, sampling_seed=20261020 + STUDENT_STEPS, learning_rate=1e-4, warmup=50, gradient_clip=1.0,
                 updates=2000, checkpoint_every=500, accumulation=1),
        selection='No automatic selection: every checkpoint saved, endpoint distance on 45 held-out prompts and their images; spelling decides',
        scope=f'{STUDENT_STEPS}-step lightning LoRA (rank 32, attention of the 30 blocks) on version 1, trajectory distillation from the teacher')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/lightning_lora' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/lightning_lora/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App(f'nihonga-phase10-lightning-{STUDENT_STEPS}')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'lightning_lora_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=131072,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls', 'trainable_parameters')}), flush=True)
    return request_key


def lightning_table(report):
    head = '| Checkpoint | Endpoint distance (all 45) | Text (20) | General (25) |\n|---|---|---|---|\n'
    display(Markdown(head + chr(10).join(f"| step {c['step']} | {c['endpoint_mean']:.4f} | {c['endpoint_text']:.4f} | {c['endpoint_general']:.4f} |"
                                         for c in report['checkpoints'])))

LAUNCH_ARGS = ['--dry-run']
RUN_L8 = 'data/phase10/lightning_lora/' + main_train()
LAUNCH_ARGS = []
summary_l8 = reuse_or_launch(RUN_L8, main_train)
if summary_l8 and summary_l8.get('status') == 'passed':
    lightning_table(summary_l8)

## 3. The 4-step lightning LoRA

The same, with one student step for every 10 teacher steps.

In [ ]:
# Lightning LoRA, 4 steps: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run step 1 first).
STUDENT_STEPS, BRIDGE_DTYPE = 4, 'bfloat16'
SOURCES['lightning_lora_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/lightning_lora') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [f'step_{s:04d}' for s in range(0, request['fit']['updates'] + 1, request['fit']['checkpoint_every'])]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # ---- Version 1 student with a trainable LoRA; trajectory distillation to K steps; every checkpoint evaluated. ----
        import random
        from diffusers import FlowMatchEulerDiscreteScheduler
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        from torch.nn.attention import SDPBackend, sdpa_kernel
        K = request['student_steps']
        stride = request['teacher_steps'] // K
        keep = list(range(0, request['teacher_steps'] + 1, stride))
        fit, lora = request['fit'], request['lora']
        outputs = {v: {} for v in variants}
        conditioned, components = {}, {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        bridge_slots = []
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(),
                                                     keep_dtype=request['bridge_dtype'] == 'float32')
            bridge_slots.append(bridge_spec['layer'])
        torch.manual_seed(fit['init_seed'])
        for slot, block in enumerate(list(model.transformer_blocks)):
            if slot in bridge_slots:
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), lora['rank'], lora['alpha']))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], lora['rank'], lora['alpha'])
            model.transformer_blocks[slot] = components['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        model.requires_grad_(False)
        model.gradient_checkpointing = False
        trainable = {n: p for n, p in model.named_parameters() if '.attn.' in n and ('.down.' in n or '.up.' in n)}
        if len(trainable) != 30 * 4 * 2:
            raise RuntimeError('Adapter structure differs.')
        for parameter in trainable.values():
            parameter.requires_grad_(True)
        report['trainable_parameters'] = sum(p.numel() for p in trainable.values())
        gc.collect()
        torch.cuda.empty_cache()
        model.eval()

        # Teacher trajectories: only the K+1 points on the student's grid are kept in memory.
        traj_dir = Path(request['trajectories']['remote_directory'])

        def load_trajectory(source_id):
            path = traj_dir / (source_id + '-trajectory.pt')
            if digest(path) != request['trajectories']['sha256'][source_id]:
                raise RuntimeError('Trajectory differs: ' + source_id)
            return torch.load(path, weights_only=True, map_location='cpu')

        train_items = []
        for source_id in request['trajectories']['train_ids']:
            item = load_trajectory(source_id)
            train_items.append({'source_id': source_id, 'latents': item['latents'][keep].contiguous(), 'sigmas': item['sigmas'][keep],
                                'timesteps': [item['timesteps'][i] for i in keep[:-1]], 'conditioning': item['conditioning']})
        held_final, grid_sigmas = {}, None
        for row in request['rows']:
            item = load_trajectory(row['source_id'])
            held_final[row['source_id']] = item['latents'][-1].float()
            grid_sigmas = item['sigmas'][keep]
        report['train_items'] = len(train_items)

        def train_loss(item, segment):
            # One student step from the teacher's latent must land where the teacher's `stride` steps land.
            x = item['latents'][segment].unsqueeze(0).to('cuda')
            x_next = item['latents'][segment + 1].unsqueeze(0).to('cuda')
            sigma, sigma_next = item['sigmas'][segment].item(), item['sigmas'][segment + 1].item()
            target = (x_next.float() - x.float()) / (sigma_next - sigma)
            cond = item['conditioning']
            prediction = model(hidden_states=x, timestep=torch.tensor([item['timesteps'][segment]], device='cuda', dtype=x.dtype),
                               encoder_hidden_states=cond['encoder_hidden_states'].to('cuda'),
                               encoder_hidden_states_mask=cond['encoder_hidden_states_mask'].to('cuda'),
                               img_shapes=cond['img_shapes'], img_mask=cond['img_mask'].to('cuda'), attention_kwargs=None,
                               kv_cache=QwenImage21KVCache(len(model.transformer_blocks)), kv_cache_mode='extract',
                               return_dict=False)[0][:, -x.shape[1]:].float()
            loss = (prediction - target).square().mean() / target.square().mean().clamp_min(1e-12)
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss.')
            return loss

        class SubsetScheduler(FlowMatchEulerDiscreteScheduler):
            # The teacher's 40-step sigma grid (same dynamic shift and terminal stretch), keeping every `stride`-th point.
            stride = 1

            def set_timesteps(self, *args, **kwargs):
                super().set_timesteps(*args, **kwargs)
                self.sigmas = self.sigmas[::self.stride].clone()
                self.timesteps = (self.sigmas[:-1] * self.config.num_train_timesteps).to(self.sigmas.device)
                self.num_inference_steps = len(self.timesteps)

        default_scheduler = pipe.scheduler
        student_scheduler = SubsetScheduler.from_config(default_scheduler.config)
        student_scheduler.stride = stride
        report['checkpoints'] = []
        report['losses'] = []
        started = time.perf_counter()

        def evaluate(step):
            # Quantitative: K-step final latent against the teacher's 40-step final latent (same noise), 45 held-out prompts.
            # Qualitative: the same 45 images (four-seed text grids and 25 general prompts).
            name = f'step_{step:04d}'
            pipe.scheduler = student_scheduler
            errors = {}
            for row in request['rows']:
                final = {}

                def on_step_end(pipeline, index, timestep, tensors):
                    if index == K - 1:
                        final['latents'] = tensors['latents'].detach().float().cpu()
                    return {}

                torch.cuda.synchronize()
                t0 = time.perf_counter()
                with torch.inference_mode():
                    image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=request['teacher_steps'],
                                 true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                                 use_kv_cache=True, output_type='pil', callback_on_step_end=on_step_end).images[0]
                torch.cuda.synchronize()
                seconds = time.perf_counter() - t0
                if (pipe.scheduler.sigmas.detach().float().cpu() - grid_sigmas).abs().max() > 1e-6:
                    raise RuntimeError('Student sigma grid differs from the teacher subset.')
                reference = held_final[row['source_id']]
                error = ((final['latents'][0] - reference).norm() / reference.norm()).item()
                errors[row['source_id']] = error
                path = directory / (row['source_id'] + '-' + name + '.png')
                image.save(path, format='PNG')
                outputs[name][row['source_id']] = {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size,
                                                   'seconds': seconds, 'endpoint_relative_l2': error}
                report['calls'][name] += 1
            pipe.scheduler = default_scheduler
            text_ids = {r['source_id'] for r in request['rows'] if r['subdimension'] in ('text_rendering', 'posters')}
            mean = lambda values: sum(values) / len(values)
            entry = dict(step=step, seconds=time.perf_counter() - started,
                         endpoint_mean=mean(list(errors.values())),
                         endpoint_text=mean([e for s, e in errors.items() if s in text_ids]),
                         endpoint_general=mean([e for s, e in errors.items() if s not in text_ids]))
            if step > 0:
                path = directory / f'lora_{K}steps_step_{step:04d}.pt'
                temporary = path.with_suffix('.tmp')
                torch.save({'request_key': key, 'step': step, 'student_steps': K, 'lora': lora, 'bridge_dtype': request['bridge_dtype'],
                            'state_dict': {n: p.detach().cpu().clone() for n, p in trainable.items()}}, temporary)
                temporary.replace(path)
                entry['checkpoint'] = dict(name=path.name, sha256=digest(path), bytes=path.stat().st_size)
                report['artifacts'].append(entry['checkpoint'])
            report['checkpoints'].append(entry)
            print(f"{K}-step LoRA, step {step}: endpoint {entry['endpoint_mean']:.4f} (text {entry['endpoint_text']:.4f}, "
                  f"general {entry['endpoint_general']:.4f}) ({entry['seconds']:.0f}s)", flush=True)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()

        optimizer = torch.optim.AdamW(list(trainable.values()), lr=fit['learning_rate'], weight_decay=0.0)
        rng = random.Random(fit['sampling_seed'])
        running = []
        for step in range(fit['updates'] + 1):
            if step % fit['checkpoint_every'] == 0:
                model.eval()
                evaluate(step)
            if step == fit['updates']:
                break
            for group in optimizer.param_groups:
                group['lr'] = fit['learning_rate'] * min(1.0, (step + 1) / fit['warmup'])
            optimizer.zero_grad(set_to_none=True)
            # Backward recomputes checkpointed blocks: keep the forward attention kernel and cache context.
            with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for _ in range(fit['accumulation']):
                    loss = train_loss(rng.choice(train_items), rng.randrange(K)) / fit['accumulation']
                    loss.backward()
                    running.append(loss.item() * fit['accumulation'])
            torch.nn.utils.clip_grad_norm_(list(trainable.values()), fit['gradient_clip'])
            optimizer.step()
            report['optimizer_updates'] += 1
            if len(running) >= 50:
                report['losses'].append({'step': step + 1, 'mean_loss': sum(running) / len(running)})
                running = []
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_train():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    traj = load(RUN + '/summary.json')
    traj_request = load(RUN + '/request.json')['request']
    healing = load(next((root / 'data/phase8/healing').glob('*/request.json')).relative_to(root))['request']
    shas = {r['source_id']: r['trajectory']['sha256'] for r in traj['rows']}
    request = {k: traj_request[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                            'canonical_tables_key', 'core_source', 'wrapper_source', 'teacher_steps')}
    request.update(
        version='phase10-lightning-lora-v1', worker_source=SOURCES['lightning_lora_worker.py'], execution_account='new', candidate_only=True,
        student_steps=STUDENT_STEPS, bridge_dtype=BRIDGE_DTYPE,
        conditioned_bridge_source=healing['conditioned_bridge_source'], components_source=healing['components_source'],
        bridges=healing['bridges'],
        trajectories=dict(remote_directory=f"/phase3/phase10/teacher_trajectories/{traj['request_key']}",
                          train_ids=[r['source_id'] for r in traj_request['rows'] if r['split'] == 'train'], sha256=shas),
        rows=[r for r in traj_request['rows'] if r['split'] == 'held_out'],
        lora=dict(rank=32, alpha=32, targets=['to_q', 'to_k', 'to_v', 'to_out.0']),
        fit=dict(init_seed=20261019, sampling_seed=20261020 + STUDENT_STEPS, learning_rate=1e-4, warmup=50, gradient_clip=1.0,
                 updates=2000, checkpoint_every=500, accumulation=1),
        selection='No automatic selection: every checkpoint saved, endpoint distance on 45 held-out prompts and their images; spelling decides',
        scope=f'{STUDENT_STEPS}-step lightning LoRA (rank 32, attention of the 30 blocks) on version 1, trajectory distillation from the teacher')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/lightning_lora' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/lightning_lora/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App(f'nihonga-phase10-lightning-{STUDENT_STEPS}')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'lightning_lora_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=131072,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls', 'trainable_parameters')}), flush=True)
    return request_key


def lightning_table(report):
    head = '| Checkpoint | Endpoint distance (all 45) | Text (20) | General (25) |\n|---|---|---|---|\n'
    display(Markdown(head + chr(10).join(f"| step {c['step']} | {c['endpoint_mean']:.4f} | {c['endpoint_text']:.4f} | {c['endpoint_general']:.4f} |"
                                         for c in report['checkpoints'])))

LAUNCH_ARGS = ['--dry-run']
RUN_L4 = 'data/phase10/lightning_lora/' + main_train()
LAUNCH_ARGS = []
summary_l4 = reuse_or_launch(RUN_L4, main_train)
if summary_l4 and summary_l4.get('status') == 'passed':
    lightning_table(summary_l4)